# Plant Leaves Super-Resolution Challenge
## 4x cGAN-based Super-Resolution (32×32 → 128×128)

**Architecture:** RRDB Generator + PatchGAN Discriminator  
**Losses:** L1 (pixel) + Perceptual (VGG19) + Adversarial  
**Metric:** MAE (lower = better) | Baseline: 17.35

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models as models

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    BASE = Path('/kaggle/input/plant-leaves-super-resolution-challenge')
    OUT_DIR = Path('/kaggle/working')
else:
    BASE = Path('/Users/sanskar/dev/NPPE2/dataset')
    OUT_DIR = Path('/Users/sanskar/dev/NPPE2')

TRAIN_HR    = BASE / 'train_High_Resolution'
TRAIN_LR    = BASE / 'train_Low_Resolution'
TEST_LR     = BASE / 'test_Low_Resolution'
VGG_WEIGHTS = BASE / 'vgg19_weights.pth'
CHECKPOINT_G = OUT_DIR / 'generator_best.pth'
CHECKPOINT_D = OUT_DIR / 'discriminator_best.pth'
SUBMISSION   = OUT_DIR / 'submission.csv'

print(f"Train HR : {len(list(TRAIN_HR.glob('*.png')))} images")
print(f"Train LR : {len(list(TRAIN_LR.glob('*.png')))} images")
print(f"Test  LR : {len(list(TEST_LR.glob('*.png')))} images")

# ── Hyper-parameters ───────────────────────────────────────────────────
CFG = dict(
    lr_size       = 32,
    hr_size       = 128,
    scale         = 4,
    # Generator
    nf            = 64,
    nb            = 16,
    gc            = 32,
    # Discriminator
    d_nf          = 64,
    # DataLoader
    batch_size    = 16,
    num_workers   = 2,
    # Phase 1 — pixel warm-up (G only)
    warmup_epochs = 100,
    warmup_lr     = 2e-4,
    # Phase 2 — full cGAN
    gan_epochs    = 80,
    gan_lr_g      = 1e-4,
    gan_lr_d      = 1e-4,
    # Phase 3 — pixel fine-tune (G only, pure Charbonnier)
    ft_epochs     = 20,
    ft_lr         = 5e-6,
    # Loss weights (Phase 2)
    lambda_adv    = 0.001,   # kept tiny — texture realism, not spatial accuracy
    # Misc
    amp           = True,
)
print(f"\nTotal training epochs : {CFG['warmup_epochs']} + {CFG['gan_epochs']} + {CFG['ft_epochs']} = "
      f"{CFG['warmup_epochs']+CFG['gan_epochs']+CFG['ft_epochs']}")
print("Config loaded ✓")

## Dataset & DataLoader

In [ ]:
class SRDataset(Dataset):
    """Paired LR/HR dataset with joint augmentation."""

    def __init__(self, lr_dir, hr_dir=None, augment=True):
        self.lr_paths = sorted(Path(lr_dir).glob('*.png'))
        self.hr_dir   = Path(hr_dir) if hr_dir else None
        self.augment  = augment

        # Normalise to [-1, 1]  (generator outputs tanh)
        self.to_tensor = T.ToTensor()   # [0,1]

    def __len__(self):
        return len(self.lr_paths)

    def _norm(self, t):
        return t * 2.0 - 1.0  # [0,1] -> [-1,1]

    def __getitem__(self, idx):
        lr_path = self.lr_paths[idx]
        lr_img  = Image.open(lr_path).convert('RGB')

        if self.hr_dir is not None:
            # Match filename: agrivision_train_XXXX.png
            hr_path = self.hr_dir / lr_path.name
            hr_img  = Image.open(hr_path).convert('RGB')

            # ── Joint augmentation ──────────────────────────
            if self.augment:
                # Horizontal flip
                if random.random() > 0.5:
                    lr_img = TF.hflip(lr_img)
                    hr_img = TF.hflip(hr_img)
                # Vertical flip
                if random.random() > 0.5:
                    lr_img = TF.vflip(lr_img)
                    hr_img = TF.vflip(hr_img)
                # 90° rotation
                k = random.randint(0, 3)
                lr_img = TF.rotate(lr_img, k * 90)
                hr_img = TF.rotate(hr_img, k * 90)

            lr_t = self._norm(self.to_tensor(lr_img))
            hr_t = self._norm(self.to_tensor(hr_img))
            return lr_t, hr_t, lr_path.name

        else:
            lr_t = self._norm(self.to_tensor(lr_img))
            return lr_t, lr_path.name


# ── Build loaders ──────────────────────────────────────────────────────
train_ds = SRDataset(TRAIN_LR, TRAIN_HR, augment=True)
train_loader = DataLoader(
    train_ds,
    batch_size  = CFG['batch_size'],
    shuffle     = True,
    num_workers = CFG['num_workers'],
    pin_memory  = True,
    drop_last   = True,
)

test_ds = SRDataset(TEST_LR, hr_dir=None, augment=False)
test_loader = DataLoader(
    test_ds,
    batch_size  = 8,
    shuffle     = False,
    num_workers = CFG['num_workers'],
    pin_memory  = True,
)

print(f"Train batches : {len(train_loader)}")
print(f"Test  samples : {len(test_ds)}")

In [ ]:
# ── Quick visual sanity check ──────────────────────────────────────────
lr_batch, hr_batch, names = next(iter(train_loader))

def denorm(t):
    """[-1,1] -> [0,1] -> uint8"""
    return ((t.clamp(-1, 1) + 1) / 2 * 255).byte()

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    lr_np = denorm(lr_batch[i]).permute(1,2,0).numpy()
    hr_np = denorm(hr_batch[i]).permute(1,2,0).numpy()
    axes[0, i].imshow(lr_np); axes[0, i].set_title(f'LR 32×32', fontsize=8); axes[0, i].axis('off')
    axes[1, i].imshow(hr_np); axes[1, i].set_title(f'HR 128×128', fontsize=8); axes[1, i].axis('off')

plt.suptitle('Training Samples (LR → HR pairs)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("Shapes — LR:", lr_batch.shape, "  HR:", hr_batch.shape)

## Model Architectures

### Generator: RRDB-Net (ESRGAN-style, trained from scratch)
- Residual-in-Residual Dense Blocks for feature extraction  
- 2× nearest-neighbour upsampling × 2 = 4× total  
- `tanh` output → pixel values in [-1, 1]

### Discriminator: PatchGAN with Spectral Normalisation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  GENERATOR — RRDB-Net with Channel Attention + PixelShuffle (scratch)
# ═══════════════════════════════════════════════════════════════════════

def icnr_init(weight, scale=2):
    """ICNR init for PixelShuffle conv: initial output ≈ nearest-neighbour.
    Eliminates checkerboard artifacts from random PixelShuffle init."""
    out_ch, in_ch, kH, kW = weight.shape
    sub_out = out_ch // (scale * scale)
    sub = torch.empty(sub_out, in_ch, kH, kW)
    nn.init.kaiming_normal_(sub, a=0.2, mode='fan_in')
    weight.data.copy_(sub.repeat_interleave(scale * scale, dim=0))


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, nf, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(nf, max(nf // reduction, 4), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(nf // reduction, 4), nf, 1, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(self.pool(x))


class DenseBlock(nn.Module):
    """Residual Dense Block with Channel Attention."""
    def __init__(self, nf=64, gc=32, res_scale=0.2):
        super().__init__()
        self.c1 = nn.Conv2d(nf,        gc, 3, 1, 1)
        self.c2 = nn.Conv2d(nf + gc,   gc, 3, 1, 1)
        self.c3 = nn.Conv2d(nf + 2*gc, gc, 3, 1, 1)
        self.c4 = nn.Conv2d(nf + 3*gc, gc, 3, 1, 1)
        self.c5 = nn.Conv2d(nf + 4*gc, nf, 3, 1, 1)
        self.ca  = ChannelAttention(nf)
        self.act = nn.LeakyReLU(0.2, inplace=True)
        self.res_scale = res_scale
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2, mode='fan_in')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x1 = self.act(self.c1(x))
        x2 = self.act(self.c2(torch.cat([x, x1], 1)))
        x3 = self.act(self.c3(torch.cat([x, x1, x2], 1)))
        x4 = self.act(self.c4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.c5(torch.cat([x, x1, x2, x3, x4], 1))
        return self.ca(x5) * self.res_scale + x


class RRDB(nn.Module):
    """Residual-in-Residual Dense Block (3 × DenseBlock)."""
    def __init__(self, nf=64, gc=32, res_scale=0.2):
        super().__init__()
        self.RDB1 = DenseBlock(nf, gc, res_scale)
        self.RDB2 = DenseBlock(nf, gc, res_scale)
        self.RDB3 = DenseBlock(nf, gc, res_scale)
        self.res_scale = res_scale

    def forward(self, x):
        out = self.RDB3(self.RDB2(self.RDB1(x)))
        return out * self.res_scale + x


class RRDBNet(nn.Module):
    """4× SR Generator: bicubic residual skip + PixelShuffle upsampling.

    Key design choices:
    - Bicubic residual: output = clamp(bicubic(LR) + net(LR), -1, 1)
      → net only learns the HIGH-FREQ residual; bicubic handles low-freq
      → dramatically reduces what the network needs to learn
    - ICNR init on PixelShuffle convs: initial output ≈ bicubic
    - Zero-init conv_last: residual starts near zero → stable early training
    """

    def __init__(self, in_nc=3, out_nc=3, nf=64, nb=16, gc=32):
        super().__init__()
        self.conv_first = nn.Conv2d(in_nc, nf, 3, 1, 1)

        self.body      = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.conv_body = nn.Conv2d(nf, nf, 3, 1, 1)

        # PixelShuffle 2× → 2× = 4× total (sub-pixel convolution)
        self.up1 = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),                     # nf*4 → nf, 2× spatial
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.conv_hr   = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_last = nn.Conv2d(nf, out_nc, 3, 1, 1)
        self.lrelu     = nn.LeakyReLU(0.2, inplace=True)

        self._init_weights()

    def _init_weights(self):
        for m in [self.conv_first, self.conv_body, self.conv_hr]:
            nn.init.kaiming_normal_(m.weight, a=0.2, mode='fan_in')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        # Zero-init residual predictor → output starts as pure bicubic baseline
        nn.init.zeros_(self.conv_last.weight)
        nn.init.zeros_(self.conv_last.bias)
        # ICNR init on PixelShuffle convs → checkerboard-free from epoch 1
        icnr_init(self.up1[0].weight, scale=2)
        icnr_init(self.up2[0].weight, scale=2)
        for module in [self.up1, self.up2]:
            for m in module.modules():
                if isinstance(m, nn.Conv2d) and m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # Bicubic baseline: provides low-freq structure for free
        bicubic = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False)
        # Network predicts the high-frequency residual only
        feat = self.conv_first(x)
        feat = feat + self.conv_body(self.body(feat))
        feat = self.up1(feat)    # 32 → 64
        feat = self.up2(feat)    # 64 → 128
        residual = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return (bicubic + residual).clamp(-1.0, 1.0)


# ═══════════════════════════════════════════════════════════════════════
#  DISCRIMINATOR — PatchGAN with Spectral Norm
# ═══════════════════════════════════════════════════════════════════════

class Discriminator(nn.Module):
    """PatchGAN (output: 14×14 patches) with spectral normalisation."""

    def __init__(self, in_nc=3, nf=64):
        super().__init__()
        sn = nn.utils.spectral_norm

        def block(ic, oc, stride=2, bn=True):
            layers = [sn(nn.Conv2d(ic, oc, 4, stride, 1, bias=not bn))]
            if bn:
                layers.append(nn.BatchNorm2d(oc))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_nc, nf,    stride=2, bn=False),  # 128→64
            *block(nf,    nf*2,  stride=2),             # 64→32
            *block(nf*2,  nf*4,  stride=2),             # 32→16
            *block(nf*4,  nf*8,  stride=1),             # 16→15
            sn(nn.Conv2d(nf*8, 1, 4, 1, 1)),            # 15→14
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)


# ── Instantiate ────────────────────────────────────────────────────────
G = RRDBNet(nf=CFG['nf'], nb=CFG['nb'], gc=CFG['gc']).to(DEVICE)
D = Discriminator(nf=CFG['d_nf']).to(DEVICE)

n_G = sum(p.numel() for p in G.parameters()) / 1e6
n_D = sum(p.numel() for p in D.parameters()) / 1e6
print(f"Generator params    : {n_G:.2f} M")
print(f"Discriminator params: {n_D:.2f} M")

with torch.no_grad():
    _lr = torch.randn(2, 3, 32, 32).to(DEVICE)
    _sr = G(_lr)
    _d  = D(_sr)
print(f"G output: {_sr.shape}  range=[{_sr.min():.3f}, {_sr.max():.3f}]")
print(f"D output: {_d.shape}")
assert _sr.shape == (2, 3, 128, 128), "G output shape mismatch!"
del _lr, _sr, _d

## Loss Functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  LOSS FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════

# ── 1. Charbonnier Loss (smooth L1, better gradient near 0) ────────────
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps2 = eps ** 2
    def forward(self, pred, target):
        return torch.sqrt((pred - target) ** 2 + self.eps2).mean()

# ── 2. FFT Frequency Loss (recovers high-frequency leaf textures) ──────
class FFTLoss(nn.Module):
    """L1 on magnitude spectrum — penalises missing fine textures."""
    def forward(self, pred, target):
        pred_fft   = torch.fft.rfft2(pred,   norm='backward')
        target_fft = torch.fft.rfft2(target, norm='backward')
        pred_mag   = torch.abs(pred_fft)
        target_mag = torch.abs(target_fft)
        return F.l1_loss(pred_mag, target_mag)

# ── 3. Gradient (Edge) Loss (preserves veins / necrotic borders) ───────
class GradientLoss(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32)
        self.register_buffer('kx', kx.view(1,1,3,3).repeat(3,1,1,1))
        self.register_buffer('ky', ky.view(1,1,3,3).repeat(3,1,1,1))

    def forward(self, pred, target):
        gx_p = F.conv2d(pred,   self.kx, padding=1, groups=3)
        gy_p = F.conv2d(pred,   self.ky, padding=1, groups=3)
        gx_t = F.conv2d(target, self.kx, padding=1, groups=3)
        gy_t = F.conv2d(target, self.ky, padding=1, groups=3)
        return F.l1_loss(gx_p, gx_t) + F.l1_loss(gy_p, gy_t)

# ── 4. VGG Perceptual Loss ─────────────────────────────────────────────
class VGGPerceptualLoss(nn.Module):
    def __init__(self, weights_path: Path):
        super().__init__()
        try:
            vgg = models.vgg19(weights=None)
        except TypeError:
            vgg = models.vgg19(pretrained=False)
        try:
            state = torch.load(str(weights_path), map_location='cpu', weights_only=False)
        except TypeError:
            state = torch.load(str(weights_path), map_location='cpu')
        vgg.load_state_dict(state)

        self.slice1 = nn.Sequential(*list(vgg.features)[:9]).eval()   # relu2_2
        self.slice2 = nn.Sequential(*list(vgg.features)[9:14]).eval() # relu3_2 (finer spatial than relu3_4)
        for p in self.parameters():
            p.requires_grad = False

        mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)
        self.register_buffer('mean', mean)
        self.register_buffer('std',  std)

    def _prep(self, x):
        x = (x.clamp(-1,1) + 1.0) / 2.0
        return (x - self.mean) / self.std

    def forward(self, pred, target):
        p, t = self._prep(pred), self._prep(target)
        f1p, f1t = self.slice1(p), self.slice1(t)
        f2p, f2t = self.slice2(f1p), self.slice2(f1t)
        return F.l1_loss(f1p, f1t) + F.l1_loss(f2p, f2t)

# ── 5. LSGAN adversarial (MSE on 0/1 labels — very stable) ────────────
adv_loss_fn = nn.MSELoss()

# ── Instantiate ────────────────────────────────────────────────────────
charb_loss_fn = CharbonnierLoss(eps=1e-3)
fft_loss_fn   = FFTLoss()
grad_loss_fn  = GradientLoss().to(DEVICE)
perc_loss_fn  = VGGPerceptualLoss(VGG_WEIGHTS).to(DEVICE)

print("All loss functions ready ✓")
print("  Charbonnier  — primary pixel loss")
print("  FFT          — frequency spectrum matching")
print("  Gradient     — edge/texture preservation")
print("  Perceptual   — VGG19 relu2_2 + relu3_2 (finer spatial)")
print("  LSGAN        — adversarial stability")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  EMA — Exponential Moving Average of Generator Weights
#  Free ~0.5-1.0 MAE improvement at zero training cost.
#  EMA weights are used ONLY for inference (more stable predictions).
# ═══════════════════════════════════════════════════════════════════════

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.model  = model
        self.decay  = decay
        self.shadow = {k: v.clone().float()
                       for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self):
        for k, v in self.model.state_dict().items():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.float()

    def apply(self):
        """Copy EMA weights into model (for inference)."""
        self.backup = {k: v.clone() for k, v in self.model.state_dict().items()}
        ema_sd = {k: v.to(next(self.model.parameters()).dtype)
                  for k, v in self.shadow.items()}
        self.model.load_state_dict(ema_sd)

    def restore(self):
        """Restore original weights (for continued training)."""
        self.model.load_state_dict(self.backup)

    def save(self, path):
        self.apply()
        torch.save(self.model.state_dict(), path)
        self.restore()

ema = EMA(G, decay=0.999)
print("EMA tracker initialised ✓  (decay=0.999)")

## Training

**Phase 1 — Pixel Warm-up (G only, pure L1)**  
The generator first learns faithful pixel-level reconstruction. This directly minimises MAE and provides a solid structural base.

**Phase 2 — cGAN Fine-tune (G + D, L1 + Perceptual + Adversarial)**  
Add texture realism while keeping the high pixel-fidelity weight on L1.

In [ ]:
# ── Optimiser & Scheduler ──────────────────────────────────────────────
opt_G_warmup = torch.optim.Adam(G.parameters(), lr=CFG['warmup_lr'],
                                betas=(0.9, 0.999), weight_decay=0)
sched_G_warmup = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_G_warmup, T_max=CFG['warmup_epochs'], eta_min=5e-6)

# ── Mixed-precision (forward-compatible with PyTorch 2.x) ──────────────
use_amp = CFG['amp'] and DEVICE.type == 'cuda'
# GradScaler: new API in 2.x, fall back to legacy on 1.x
try:
    scaler = torch.amp.GradScaler(enabled=use_amp)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ── autocast helper ────────────────────────────────────────────────────
import contextlib
def autocast():
    """Forward-compatible autocast context (works on both CUDA and CPU)."""
    if use_amp:
        return torch.amp.autocast(device_type=DEVICE.type)
    return contextlib.nullcontext()

# ── MAE helper (matches leaderboard metric exactly) ────────────────────
def batch_mae(pred, target):
    p = ((pred.detach().clamp(-1,1) + 1) / 2 * 255)
    t = ((target.detach().clamp(-1,1) + 1) / 2 * 255)
    return (p - t).abs().mean().item()

history = {'epoch': [], 'charb': [], 'fft': [], 'grad': [], 'mae': []}

# ═══════════════════════════════════════════════════════════════════════
#  PHASE 1 — PIXEL WARM-UP
#  Loss = Charbonnier×100 + FFT×0.1 + Gradient×1
#  No discriminator — pure reconstruction.
# ═══════════════════════════════════════════════════════════════════════
print("=" * 65)
print(f"  Phase 1: Multi-loss Pixel Warm-up  ({CFG['warmup_epochs']} epochs)")
print("  Loss = Charbonnier×100  +  FFT×0.1  +  Gradient×1")
print("=" * 65)

best_mae = float('inf')
CKPT_EMA = OUT_DIR / 'ema_best.pth'

for epoch in range(1, CFG['warmup_epochs'] + 1):
    G.train()
    e_charb = e_fft = e_grad = e_mae = 0.0

    for lr_imgs, hr_imgs, _ in train_loader:
        lr_imgs = lr_imgs.to(DEVICE, non_blocking=True)
        hr_imgs = hr_imgs.to(DEVICE, non_blocking=True)

        opt_G_warmup.zero_grad(set_to_none=True)
        with autocast():
            sr      = G(lr_imgs)
            l_charb = charb_loss_fn(sr, hr_imgs)
            l_fft   = fft_loss_fn(sr, hr_imgs)
            l_grad  = grad_loss_fn(sr, hr_imgs)
            loss    = 100.0 * l_charb + 0.1 * l_fft + 1.0 * l_grad

        scaler.scale(loss).backward()
        scaler.unscale_(opt_G_warmup)
        nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        scaler.step(opt_G_warmup)
        scaler.update()

        ema.update()

        e_charb += l_charb.item()
        e_fft   += l_fft.item()
        e_grad  += l_grad.item()
        e_mae   += batch_mae(sr, hr_imgs)

    sched_G_warmup.step()

    n = len(train_loader)
    epoch_mae = e_mae / n
    history['epoch'].append(epoch)
    history['charb'].append(e_charb / n)
    history['fft'].append(e_fft / n)
    history['grad'].append(e_grad / n)
    history['mae'].append(epoch_mae)

    if epoch_mae < best_mae:
        best_mae = epoch_mae
        torch.save(G.state_dict(), CHECKPOINT_G)
        ema.save(CKPT_EMA)

    if epoch % 10 == 0 or epoch == 1:
        lr_now = opt_G_warmup.param_groups[0]['lr']
        print(f"[{epoch:3d}/{CFG['warmup_epochs']}] "
              f"Charb={e_charb/n:.4f}  FFT={e_fft/n:.4f}  "
              f"Grad={e_grad/n:.4f}  MAE={epoch_mae:.4f}  "
              f"best={best_mae:.4f}  lr={lr_now:.2e}")

print(f"\nPhase 1 done.  Best MAE: {best_mae:.4f}")
G.load_state_dict(torch.load(CHECKPOINT_G, map_location=DEVICE))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  PHASE 2 — FULL cGAN FINE-TUNE
#  G Loss = Charb×100 + FFT×0.1 + Grad×1 + Perc×1 + Adv×lambda_adv
#  D Loss = LSGAN
# ═══════════════════════════════════════════════════════════════════════
print("=" * 65)
print(f"  Phase 2: cGAN Fine-tune  ({CFG['gan_epochs']} epochs)")
print(f"  G = Charb×100 + FFT×0.1 + Grad×1 + Perc×1 + Adv×{CFG['lambda_adv']}")
print("=" * 65)

opt_G = torch.optim.Adam(G.parameters(), lr=CFG['gan_lr_g'],
                         betas=(0.9, 0.999), weight_decay=0)
opt_D = torch.optim.Adam(D.parameters(), lr=CFG['gan_lr_d'],
                         betas=(0.9, 0.999), weight_decay=0)

sched_G = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_G, T_max=CFG['gan_epochs'], eta_min=5e-6)
sched_D = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_D, T_max=CFG['gan_epochs'], eta_min=5e-6)

try:
    scaler2 = torch.amp.GradScaler(enabled=use_amp)
except TypeError:
    scaler2 = torch.cuda.amp.GradScaler(enabled=use_amp)

history2 = {k: [] for k in
            ['epoch','g_total','g_charb','g_fft','g_grad',
             'g_perc','g_adv','d_loss','mae']}

for epoch in range(1, CFG['gan_epochs'] + 1):
    G.train(); D.train()
    e_g = e_charb = e_fft = e_grad = e_perc = e_adv = 0.0
    e_d = e_mae = 0.0

    for lr_imgs, hr_imgs, _ in train_loader:
        lr_imgs = lr_imgs.to(DEVICE, non_blocking=True)
        hr_imgs = hr_imgs.to(DEVICE, non_blocking=True)

        # ── Train Discriminator ──────────────────────────────────────
        opt_D.zero_grad(set_to_none=True)
        with autocast():
            with torch.no_grad():
                sr_d = G(lr_imgs)
            d_real = D(hr_imgs)
            d_fake = D(sr_d.detach())
            d_loss = (adv_loss_fn(d_real, torch.ones_like(d_real)) +
                      adv_loss_fn(d_fake, torch.zeros_like(d_fake))) * 0.5

        scaler2.scale(d_loss).backward()
        scaler2.unscale_(opt_D)
        nn.utils.clip_grad_norm_(D.parameters(), 1.0)
        scaler2.step(opt_D)
        scaler2.update()

        # ── Train Generator ──────────────────────────────────────────
        opt_G.zero_grad(set_to_none=True)
        with autocast():
            sr       = G(lr_imgs)
            d_fake_g = D(sr)
            l_charb  = charb_loss_fn(sr, hr_imgs)
            l_fft    = fft_loss_fn(sr, hr_imgs)
            l_grad   = grad_loss_fn(sr, hr_imgs)
            l_perc   = perc_loss_fn(sr, hr_imgs)
            l_adv    = adv_loss_fn(d_fake_g, torch.ones_like(d_fake_g))
            g_loss   = (100.0 * l_charb + 0.1 * l_fft +
                          1.0 * l_grad  + 1.0 * l_perc +
                        CFG['lambda_adv'] * l_adv)

        scaler2.scale(g_loss).backward()
        scaler2.unscale_(opt_G)
        nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        scaler2.step(opt_G)
        scaler2.update()

        ema.update()

        e_g     += g_loss.item()
        e_charb += l_charb.item(); e_fft  += l_fft.item()
        e_grad  += l_grad.item();  e_perc += l_perc.item()
        e_adv   += l_adv.item();   e_d    += d_loss.item()
        e_mae   += batch_mae(sr, hr_imgs)

    sched_G.step(); sched_D.step()
    n = len(train_loader)
    epoch_mae = e_mae / n

    history2['epoch'].append(epoch)
    history2['g_total'].append(e_g / n)
    history2['g_charb'].append(e_charb / n)
    history2['g_fft'].append(e_fft / n)
    history2['g_grad'].append(e_grad / n)
    history2['g_perc'].append(e_perc / n)
    history2['g_adv'].append(e_adv / n)
    history2['d_loss'].append(e_d / n)
    history2['mae'].append(epoch_mae)

    if epoch_mae < best_mae:
        best_mae = epoch_mae
        torch.save(G.state_dict(), CHECKPOINT_G)
        torch.save(D.state_dict(), CHECKPOINT_D)
        ema.save(CKPT_EMA)

    if epoch % 10 == 0 or epoch == 1:
        print(f"[{epoch:3d}/{CFG['gan_epochs']}]  "
              f"G={e_g/n:.3f}  "
              f"(C={e_charb/n:.3f} F={e_fft/n:.3f} "
              f"Gr={e_grad/n:.3f} P={e_perc/n:.3f} "
              f"A={e_adv/n:.4f})  "
              f"D={e_d/n:.4f}  MAE={epoch_mae:.4f}  best={best_mae:.4f}")

print(f"\nPhase 2 done.  Overall best MAE (train batches): {best_mae:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  PHASE 3 — PIXEL FINE-TUNE  (G only, pure Charbonnier, very low LR)
#  Recovers the MAE degraded by adversarial gradients in Phase 2.
#  EMA shadow is RESET to Phase 2 best before fine-tuning to prevent
#  the stale Phase 2 shadow from diluting Phase 3 corrections.
# ═══════════════════════════════════════════════════════════════════════
print("=" * 65)
print(f"  Phase 3: Pixel Fine-tune  ({CFG['ft_epochs']} epochs,  lr={CFG['ft_lr']:.0e})")
print("  G only — pure Charbonnier×100  (EMA shadow reset)")
print("=" * 65)

# ── Load Phase 2 best EMA into G, then sync shadow ────────────────────
try:
    _p2_state = torch.load(CKPT_EMA, map_location=DEVICE)
except TypeError:
    _p2_state = torch.load(str(CKPT_EMA), map_location=DEVICE)
G.load_state_dict(_p2_state)

# Reset EMA shadow so Phase 3 corrections are fully absorbed from step 1
ema.shadow = {k: v.clone().float() for k, v in G.state_dict().items()}

opt_ft = torch.optim.Adam(G.parameters(), lr=CFG['ft_lr'], betas=(0.9, 0.999))
sched_ft = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_ft, T_max=CFG['ft_epochs'], eta_min=1e-6)
try:
    scaler_ft = torch.amp.GradScaler(enabled=use_amp)
except TypeError:
    scaler_ft = torch.cuda.amp.GradScaler(enabled=use_amp)

CKPT_EMA_FT = OUT_DIR / 'ema_ft.pth'
best_mae_ft  = float('inf')
history3     = {'epoch': [], 'charb': [], 'mae': []}

for epoch in range(1, CFG['ft_epochs'] + 1):
    G.train()
    e_charb = e_mae = 0.0

    for lr_imgs, hr_imgs, _ in train_loader:
        lr_imgs = lr_imgs.to(DEVICE, non_blocking=True)
        hr_imgs = hr_imgs.to(DEVICE, non_blocking=True)

        opt_ft.zero_grad(set_to_none=True)
        with autocast():
            sr   = G(lr_imgs)
            loss = 100.0 * charb_loss_fn(sr, hr_imgs)

        scaler_ft.scale(loss).backward()
        scaler_ft.unscale_(opt_ft)
        nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        scaler_ft.step(opt_ft)
        scaler_ft.update()
        ema.update()

        e_charb += loss.item() / 100.0    # store raw Charbonnier
        e_mae   += batch_mae(sr, hr_imgs)

    sched_ft.step()
    n = len(train_loader)
    epoch_mae = e_mae / n

    history3['epoch'].append(epoch)
    history3['charb'].append(e_charb / n)
    history3['mae'].append(epoch_mae)

    if epoch_mae < best_mae_ft:
        best_mae_ft = epoch_mae
        ema.save(CKPT_EMA_FT)
    if best_mae_ft < best_mae:
        best_mae = best_mae_ft

    if epoch % 5 == 0 or epoch == 1:
        lr_now = opt_ft.param_groups[0]['lr']
        print(f"[{epoch:3d}/{CFG['ft_epochs']}] "
              f"Charb={e_charb/n:.4f}  MAE={epoch_mae:.4f}  "
              f"best={best_mae_ft:.4f}  lr={lr_now:.2e}")

print(f"\nPhase 3 done.  Best MAE: {best_mae_ft:.4f}")

# Load best Phase 3 EMA for inference (fall back to Phase 2 if no improvement)
try:
    ft_state = torch.load(CKPT_EMA_FT, map_location=DEVICE)
except (FileNotFoundError, TypeError):
    ft_state = torch.load(CKPT_EMA, map_location=DEVICE)
G.load_state_dict(ft_state)
G.eval()
print("Best EMA generator (Phase 3) loaded for inference ✓")

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Phase 1 losses
axes[0].plot(history['epoch'], history['charb'], label='Charbonnier', linewidth=1.5)
axes[0].plot(history['epoch'], history['fft'],   label='FFT',         linewidth=1.5)
axes[0].plot(history['epoch'], history['grad'],  label='Gradient',    linewidth=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Phase 1 — Pixel Losses'); axes[0].legend(); axes[0].grid(True)

# Phase 2 losses
axes[1].plot(history2['epoch'], history2['g_charb'], label='Charbonnier', linewidth=1.5)
axes[1].plot(history2['epoch'], history2['g_perc'],  label='Perceptual',  linewidth=1.5)
axes[1].plot(history2['epoch'], history2['d_loss'],  label='D',           linewidth=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Phase 2 — cGAN Losses'); axes[1].legend(); axes[1].grid(True)

# MAE across all 3 phases
e1 = history['epoch']
e2 = [e + CFG['warmup_epochs'] for e in history2['epoch']]
e3 = [e + CFG['warmup_epochs'] + CFG['gan_epochs'] for e in history3['epoch']]
all_mae = history['mae'] + history2['mae'] + history3['mae']

axes[2].plot(e1 + e2 + e3, all_mae, 'r-', linewidth=1.5, label='Train MAE')
axes[2].axhline(y=17.35, color='gray',   linestyle='--', alpha=0.7, label='Baseline 17.35')
axes[2].axvline(x=CFG['warmup_epochs'],
                color='orange', linestyle=':', alpha=0.7, label='Phase 2 start')
axes[2].axvline(x=CFG['warmup_epochs'] + CFG['gan_epochs'],
                color='green',  linestyle=':', alpha=0.7, label='Phase 3 start')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MAE (0-255)')
axes[2].set_title('MAE — All Phases (EMA weights)'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Best overall MAE: {best_mae:.4f}  (baseline: 17.35)")

## Visual Quality Check

In [ ]:
G.eval()

# Show LR → SR → HR for 6 training examples
check_ds = SRDataset(TRAIN_LR, TRAIN_HR, augment=False)
indices  = random.sample(range(len(check_ds)), 6)

fig, axes = plt.subplots(6, 3, figsize=(9, 18))
cols = ['LR (32×32)', 'SR (Generated 128×128)', 'HR (Ground Truth 128×128)']
for ax, col in zip(axes[0], cols):
    ax.set_title(col, fontsize=9, fontweight='bold')

for row, idx in enumerate(indices):
    lr_t, hr_t, fname = check_ds[idx]
    with torch.no_grad():
        sr_t = G(lr_t.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()

    lr_np = denorm(lr_t).permute(1,2,0).numpy()
    sr_np = denorm(sr_t).permute(1,2,0).numpy()
    hr_np = denorm(hr_t).permute(1,2,0).numpy()

    mae_val = float(np.abs(sr_np.astype(float) - hr_np.astype(float)).mean())

    axes[row, 0].imshow(lr_np); axes[row, 0].axis('off')
    axes[row, 1].imshow(sr_np); axes[row, 1].axis('off')
    axes[row, 1].set_xlabel(f'MAE={mae_val:.2f}', fontsize=8)
    axes[row, 2].imshow(hr_np); axes[row, 2].axis('off')

plt.suptitle('SR Quality Check (best checkpoint)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'quality_check.png'), dpi=150, bbox_inches='tight')
plt.show()

## Inference & Submission Generation

In [ ]:
G.eval()

rows = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Generating predictions'):
        lr_imgs, filenames = batch
        lr_imgs = lr_imgs.to(DEVICE, non_blocking=True)

        # ── Test-Time Augmentation (TTA): 8 geometries ───────────────
        # Average over: original + h-flip + v-flip + 3×rot + rot+hflip + rot+vflip
        preds = []
        for hflip in [False, True]:
            for vflip in [False, True]:
                for rot in [0, 90, 180, 270]:
                    # Apply transform
                    x = lr_imgs.clone()
                    if hflip:
                        x = torch.flip(x, dims=[3])
                    if vflip:
                        x = torch.flip(x, dims=[2])
                    if rot > 0:
                        k = rot // 90
                        x = torch.rot90(x, k, dims=[2, 3])

                    sr = G(x)

                    # Inverse transform
                    if rot > 0:
                        k = (4 - rot // 90) % 4
                        sr = torch.rot90(sr, k, dims=[2, 3])
                    if vflip:
                        sr = torch.flip(sr, dims=[2])
                    if hflip:
                        sr = torch.flip(sr, dims=[3])

                    preds.append(sr)

        sr_avg = torch.stack(preds, dim=0).mean(dim=0)   # average ensemble

        # ── Denormalise to uint8 ──────────────────────────────────────
        sr_uint8 = ((sr_avg.clamp(-1, 1) + 1) / 2 * 255).round().byte()
        # sr_uint8: (B, 3, 128, 128)  dtype=torch.uint8

        for i, fname in enumerate(filenames):
            # Flatten in row-major C-order: R1,G1,B1,R2,G2,B2,...
            # tensor shape (3, H, W) → permute → (H, W, 3) → flatten
            img_np = sr_uint8[i].permute(1, 2, 0).cpu().numpy()   # (128,128,3) uint8
            flat   = img_np.flatten()                               # 49152
            pixels = ' '.join(map(str, flat.tolist()))
            rows.append({'Id': fname, 'Pixels': pixels})

df_sub = pd.DataFrame(rows, columns=['Id', 'Pixels'])
df_sub.to_csv(SUBMISSION, index=False)

print(f"Submission saved: {SUBMISSION}")
print(f"Rows: {len(df_sub)}")
print(f"Expected pixels per row: {128*128*3} = {128*128*3 == len(df_sub['Pixels'][0].split())}")
df_sub.head(3)

In [ ]:
# ── Final validation of submission format ──────────────────────────────
df_check = pd.read_csv(SUBMISSION)
assert list(df_check.columns) == ['Id', 'Pixels'], "Wrong columns!"
assert len(df_check) == 495, f"Expected 495 rows, got {len(df_check)}"

pixel_counts = df_check['Pixels'].apply(lambda x: len(x.split()))
assert (pixel_counts == 49152).all(), \
    f"Pixel count mismatch! Min={pixel_counts.min()}, Max={pixel_counts.max()}"

# Check all values in [0, 255]
sample = df_check['Pixels'].iloc[0].split()
vals   = [int(v) for v in sample]
assert min(vals) >= 0 and max(vals) <= 255, \
    f"Values out of range! min={min(vals)}, max={max(vals)}"

print("✓  submission.csv format validation PASSED")
print(f"   Rows   : {len(df_check)}")
print(f"   Pixels : {pixel_counts.iloc[0]} per row (expected 49152)")
print(f"   Range  : [{min(vals)}, {max(vals)}]")
print(f"\nFile: {SUBMISSION}")